# 03. Эталон: правило `classify_dot` (как в `dot_classify.ipynb`)

Вместо sklearn здесь те же **функции** `missing_dist` и `classify_dot`: вода по `elevation < 0`, классы по порогам расстояния до берега. Если в parquet нет столбца `elevation`, он восстанавливается из `is_land` (суша ≥ 0, вода < 0), чтобы ветвление совпало с исходной ноутбучной логикой.

In [ ]:
from pathlib import Path
import sys
import os

for _p in (Path.cwd(), Path.cwd().parent):
    if (_p / "src" / "helpers.py").is_file():
        sys.path.insert(0, str((_p / "src").resolve()))
        os.chdir(_p)
        break

In [ ]:
from helpers import init_notebook_path, ensure_dirs, PROJECT_ROOT, PLOT_DIR, DATA_DIR, RANDOM_STATE
ROOT = init_notebook_path()
ensure_dirs()
print("PROJECT_ROOT", PROJECT_ROOT)

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from helpers import (
    GLOBAL_DATASET_PATH,
    DATA_DIR,
    classification_metrics_dict,
    print_report,
    append_metrics_store,
    PLOT_DIR,
    CLASS_NAMES,
    RANDOM_STATE,
)

In [ ]:
df = pd.read_parquet(GLOBAL_DATASET_PATH)
if 'elevation' not in df.columns:

    df = df.copy()
    df['elevation'] = np.where(df['is_land'] >= 0.5, 1.0, -1.0)
le = pickle.load(open(DATA_DIR / 'label_encoder.pkl', 'rb'))
y_all = le.transform(df['target_class'])

In [ ]:
pred_str = test_df.apply(classify_dot, axis=1)
y_pred = le.transform(pred_str)
m = classification_metrics_dict(y_test, y_pred)
print(m)
print_report(y_all, y_pred)

In [ ]:
append_metrics_store('linear_classify_dot', m)
cm = confusion_matrix(y_test, y_pred, labels=np.arange(len(CLASS_NAMES)))

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Эталон: classify_dot (dot_classify)')
plt.tight_layout(); plt.savefig(PLOT_DIR / 'confusion_linear.png', dpi=150)
plt.show()
plt.close()